# MIT805 Group Project - Group 19
## Part 1 - Exploratory Data Analysis

### Imports and Setup

In [1]:
# Colab setup

#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
# Imports

import os
import json
import math
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, TimestampType)


In [4]:
# Paths

PROJECT_ROOT = Path.cwd()

RAW_DIR   = PROJECT_ROOT / "data" / "working"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned_monthly"
FIG_DIR   = PROJECT_ROOT / "figures"
OUT_DIR   = PROJECT_ROOT / "output"

FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder
         .appName("MIT805 Part1 EDA")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "24")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

STATS = {}

print("project:", PROJECT_ROOT)
print("Spark  :", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 21:51:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


project: /Users/shreyabharat/Projects/MIT805-GroupProject
Spark  : 4.2.0


In [5]:
# Figure style

BLUE = "#0000ff"
GREEN = "#00963f"
RED = "#ff0000"
GREY =  "#52514e"

plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.axisbelow": True
})

def thousands(x, _=None):
    if x >= 1e6: return f"{x/1e6:.0f}M"
    if x >= 1e3: return f"{x/1e3:.0f}k"
    return f"{x:.0f}"

def save(fig, name, note):
    fig.text(0.02, -0.04, note, fontsize=7, color="#898781")
    fig.savefig(FIG_DIR / name)
    plt.close(fig)
    print("saved:", name)

### Data loading

In [8]:
raw_files = sorted(RAW_DIR.glob("yellow_tripdata_*.parquet"))
month_dirs = sorted(d for d in CLEAN_DIR.glob("yellow_tripdata_*_cleaned")
                    if any(d.glob("*.parquet")))

months = [d.name.replace("yellow_tripdata_","").replace("_cleaned","")
          for d in month_dirs]
paths = [str(d) for d in month_dirs]

print(f"downloaded: {len(raw_files)} months")
print(f"cleaned: {len(months)} months, {months[0]} to {months[-1]}")

missing_data = len(raw_files) - len(months)
if missing_data:
    print(f"{missing_data} downloaded data files that did not have clean rows")

STATS["months_total_raw"] = len(raw_files)
STATS["months_with_data"] = len(months)
STATS["months_no_output"] = missing_data
STATS["partial_run"] = bool(missing_data)

SCHEMA = StructType([
    StructField("tpep_pickup_datetime",TimestampType()),
    StructField("trip_distance",DoubleType()),
    StructField("pulocationid",IntegerType()),
    StructField("fare_amount",DoubleType()),
    StructField("tip_amount",DoubleType()),
    StructField("total_amount",DoubleType()),
    StructField("passenger_count",IntegerType()),
    StructField("trip_duration_min",DoubleType()),
    StructField("pickup_year",IntegerType()),
    StructField("pickup_month",IntegerType()),
    StructField("pickup_hour",IntegerType()),
    StructField("pickup_dayofweek",IntegerType()),
])

df = spark.read.schema(SCHEMA).parquet(*paths)
n = df.count()

processing_gb = sum(f.stat().st_size for p in paths for f in Path(p).rglob("*.parquet")) / 1024**3

raw_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in raw_files)

STATS.update(
    analysis_rows=int(n),
    processing_gb=round(processing_gb, 2),
    raw_rows_total=int(raw_rows),
    cleaned_rows_total=int(n),
    retention_pct_overall=round(n / raw_rows * 100, 2),
    working_set_gb=round(sum(f.stat().st_size for f in raw_files) / 1024**3, 2)
    )

print(f"retention: {STATS['retention_pct_overall']}% of {raw_rows:,} raw rows")
print(f"processing tier: {processing_gb:.2f} GB ")

downloaded: 139 months
cleaned: 139 months, 2014-06 to 2025-12
retention: 95.85% of 895,333,567 raw rows
processing tier: 18.77 GB 


### Data quality